# E08a — Data Streams and Continual Learning

**COMPSCI 713 — Week 11: Sustainability in AI**

---

## Overview

Traditional machine learning assumes data is **static** — you collect a dataset, train a model, deploy it. But the real world is a **stream**: data arrives continuously, distributions shift, and models must adapt without forgetting what they already know.

In this lesson you will:
- Understand data streams and concept drift
- Detect drift using statistical tests (ADWIN, DDM)
- Implement incremental/online learning
- Understand catastrophic forgetting in neural networks
- Apply continual learning strategies: EWC, replay, progressive networks
- Connect to sustainability: why continual learning reduces compute

### COMPSCI 713 Alignment
- **Week 11 Thursday:** Sustainability in AI: Data Streams/C-Learning (data streams, incremental/lifelong/continual vs batch learning)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## 1. Data Streams

A **data stream** is a potentially infinite sequence of data instances arriving over time.

Key challenges:
- **Volume:** cannot store all data
- **Velocity:** must process each instance quickly
- **Concept drift:** the underlying distribution P(y|x) changes over time

### Types of Concept Drift

```
Sudden drift:    ─────────╗
                          ╚─────────

Gradual drift:   ─────────╲
                            ╲───────

Incremental:     ─────────────────── (slow continuous change)

Recurring:       ─╗  ╔─╗  ╔─╗  ╔──  (seasonal patterns)
                   ╚──╝  ╚──╝  ╚─
```

In [ ]:
# ── Simulate a data stream with concept drift ──────────────────────────────

np.random.seed(42)
n_samples = 2000
drift_point = 1000

# Before drift: class 0 centred at (-1, -1), class 1 at (1, 1)
X_before = np.vstack([
    np.random.randn(drift_point//2, 2) + np.array([-1, -1]),
    np.random.randn(drift_point//2, 2) + np.array([1, 1])
])
y_before = np.array([0]*(drift_point//2) + [1]*(drift_point//2))

# After drift: classes SWAP positions
X_after = np.vstack([
    np.random.randn((n_samples-drift_point)//2, 2) + np.array([1, 1]),   # class 0 moved
    np.random.randn((n_samples-drift_point)//2, 2) + np.array([-1, -1])  # class 1 moved
])
y_after = np.array([0]*((n_samples-drift_point)//2) + [1]*((n_samples-drift_point)//2))

X_stream = np.vstack([X_before, X_after])
y_stream = np.hstack([y_before, y_after])

# Shuffle within each segment (but keep segments separate)
idx_b = np.random.permutation(drift_point)
idx_a = np.random.permutation(n_samples - drift_point)
X_stream[:drift_point] = X_stream[:drift_point][idx_b]
y_stream[:drift_point] = y_stream[:drift_point][idx_b]
X_stream[drift_point:] = X_stream[drift_point:][idx_a]
y_stream[drift_point:] = y_stream[drift_point:][idx_a]

# Visualise the drift
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, start, end, title in [
    (axes[0], 0, drift_point, 'Before Drift (t=0–1000)'),
    (axes[1], drift_point, n_samples, 'After Drift (t=1000–2000)')
]:
    X_seg, y_seg = X_stream[start:end], y_stream[start:end]
    ax.scatter(X_seg[y_seg==0, 0], X_seg[y_seg==0, 1], alpha=0.3, label='Class 0', s=10)
    ax.scatter(X_seg[y_seg==1, 0], X_seg[y_seg==1, 1], alpha=0.3, label='Class 1', s=10)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle('Concept Drift: Class distributions swap at t=1000', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Online Learning vs Batch Learning

| Aspect | Batch Learning | Online/Incremental Learning |
|---|---|---|
| Data access | Full dataset at once | One instance at a time |
| Memory | O(n) | O(1) |
| Adapts to drift? | ❌ (needs retraining) | ✅ (naturally) |
| Compute | High (full retraining) | Low (incremental updates) |
| Stability | High | Lower (sensitive to noise) |

In [ ]:
# ── Compare: Batch vs Online learning under drift ──────────────────────────

window_size = 100  # evaluate accuracy on last 100 samples

# Batch model: trained on first 500 samples, never updated
from sklearn.linear_model import LogisticRegression
batch_model = LogisticRegression()
batch_model.fit(X_stream[:500], y_stream[:500])

# Online model: updated with each new sample
online_model = SGDClassifier(loss='log_loss', learning_rate='constant',
                              eta0=0.01, random_state=42)
online_model.partial_fit(X_stream[:2], y_stream[:2], classes=[0, 1])

batch_accs, online_accs = [], []
eval_points = range(window_size, n_samples, 10)

for t in eval_points:
    X_win = X_stream[t-window_size:t]
    y_win = y_stream[t-window_size:t]

    batch_accs.append(accuracy_score(y_win, batch_model.predict(X_win)))
    online_accs.append(accuracy_score(y_win, online_model.predict(X_win)))

    # Update online model
    online_model.partial_fit(X_stream[t:t+10], y_stream[t:t+10])

plt.figure(figsize=(10, 4))
plt.plot(list(eval_points), batch_accs,  label='Batch (static)', color='red',   linewidth=2)
plt.plot(list(eval_points), online_accs, label='Online (adaptive)', color='green', linewidth=2)
plt.axvline(drift_point, color='black', linestyle='--', label='Concept drift')
plt.xlabel('Time step')
plt.ylabel('Accuracy (window)')
plt.title('Batch vs Online Learning Under Concept Drift')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Drift Detection — ADWIN

**ADWIN** (Adaptive Windowing) detects drift by maintaining a sliding window and testing whether the mean of the first half differs significantly from the second half.

In [ ]:
class ADWIN:
    """Simplified ADWIN drift detector."""

    def __init__(self, delta=0.002):
        self.delta  = delta
        self.window = deque()
        self.drift_detected = False

    def add_element(self, value):
        self.window.append(value)
        self.drift_detected = self._detect_change()

    def _detect_change(self):
        n = len(self.window)
        if n < 20:
            return False
        arr = np.array(self.window)
        # Test all split points
        for split in range(10, n-10):
            w0, w1 = arr[:split], arr[split:]
            m0, m1 = w0.mean(), w1.mean()
            n0, n1 = len(w0), len(w1)
            # Hoeffding bound
            epsilon = np.sqrt((1/(2*n0) + 1/(2*n1)) * np.log(4*n/self.delta))
            if abs(m0 - m1) > epsilon:
                # Drift detected — shrink window to recent data
                for _ in range(split):
                    self.window.popleft()
                return True
        return False


# Simulate accuracy stream and detect drift
detector = ADWIN(delta=0.002)
drift_detections = []

# Use batch model accuracy as the stream
acc_stream = []
for t in range(0, n_samples, 10):
    X_win = X_stream[t:t+10]
    y_win = y_stream[t:t+10]
    acc = accuracy_score(y_win, batch_model.predict(X_win))
    acc_stream.append((t, acc))
    detector.add_element(acc)
    if detector.drift_detected:
        drift_detections.append(t)

times, accs = zip(*acc_stream)
plt.figure(figsize=(10, 4))
plt.plot(times, accs, color='steelblue', alpha=0.7, label='Accuracy')
plt.axvline(drift_point, color='black', linestyle='--', label='True drift')
for d in drift_detections[:3]:  # show first 3 detections
    plt.axvline(d, color='red', linestyle=':', alpha=0.7, label='ADWIN detection' if d == drift_detections[0] else '')
plt.xlabel('Time step')
plt.ylabel('Accuracy')
plt.title('ADWIN Drift Detection')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Drift detections at: {drift_detections[:5]}")

## 4. Catastrophic Forgetting in Neural Networks

When a neural network is trained sequentially on tasks A then B, it tends to **forget** task A — this is **catastrophic forgetting** (McCloskey & Cohen, 1989).

This is the core challenge of **continual learning** (also called lifelong learning or incremental learning).

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim

    def make_task_data(n=200, offset=(0,0), seed=0):
        np.random.seed(seed)
        X = np.random.randn(n, 2)
        y = (X[:, 0] + offset[0] > X[:, 1] + offset[1]).astype(int)
        return torch.FloatTensor(X), torch.LongTensor(y)

    X_A, y_A = make_task_data(offset=(0, 0), seed=1)
    X_B, y_B = make_task_data(offset=(1, -1), seed=2)

    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 32), nn.ReLU(),
                nn.Linear(32, 32), nn.ReLU(),
                nn.Linear(32, 2)
            )
        def forward(self, x): return self.net(x)

    def train_task(model, X, y, epochs=100, lr=0.01):
        opt = optim.Adam(model.parameters(), lr=lr)
        for _ in range(epochs):
            opt.zero_grad()
            nn.CrossEntropyLoss()(model(X), y).backward()
            opt.step()

    def eval_acc(model, X, y):
        with torch.no_grad():
            return (model(X).argmax(1) == y).float().mean().item()

    model = MLP()
    train_task(model, X_A, y_A, epochs=200)
    acc_A_before = eval_acc(model, X_A, y_A)

    train_task(model, X_B, y_B, epochs=200)
    acc_A_after = eval_acc(model, X_A, y_A)
    acc_B_after = eval_acc(model, X_B, y_B)

    print("=== Catastrophic Forgetting Demo ===")
    print(f"Task A accuracy BEFORE training on B: {acc_A_before:.2%}")
    print(f"Task A accuracy AFTER  training on B: {acc_A_after:.2%}  ← FORGETTING")
    print(f"Task B accuracy after training on B:  {acc_B_after:.2%}")

except ImportError:
    print("PyTorch not available. Catastrophic forgetting occurs when a neural network")
    print("is fine-tuned on new data — it overwrites weights needed for old tasks.")

## 5. Continual Learning Strategies

| Strategy | Approach | Pros | Cons |
|---|---|---|---|
| **Naive fine-tuning** | Just train on new data | Simple | Catastrophic forgetting |
| **Replay / Experience Replay** | Store old samples, mix with new | Simple, effective | Memory cost |
| **EWC** (Elastic Weight Consolidation) | Penalise changes to important weights | No memory needed | Approximation |
| **Progressive Networks** | Add new columns for new tasks | No forgetting | Grows with tasks |
| **PackNet** | Prune + freeze weights per task | Efficient | Fixed capacity |

### EWC — Elastic Weight Consolidation (Kirkpatrick et al., 2017)

EWC adds a regularisation term that penalises changes to weights that were **important for previous tasks**:

$$\mathcal{L}_{EWC} = \mathcal{L}_B(\theta) + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta^*_{A,i})^2$$

- $F_i$: Fisher information — how important weight i was for task A
- $\theta^*_A$: optimal weights after task A
- $\lambda$: regularisation strength

In [ ]:
try:
    import torch
    import torch.nn as nn

    class EWC:
        """Elastic Weight Consolidation."""

        def __init__(self, model, dataset, n_samples=200):
            self.model = model
            self.params = {n: p.clone().detach() for n, p in model.named_parameters()}
            self.fisher = self._compute_fisher(dataset, n_samples)

        def _compute_fisher(self, dataset, n_samples):
            X, y = dataset
            fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters()}
            self.model.eval()
            for i in range(min(n_samples, len(X))):
                self.model.zero_grad()
                out = self.model(X[i:i+1])
                loss = nn.CrossEntropyLoss()(out, y[i:i+1])
                loss.backward()
                for n, p in self.model.named_parameters():
                    if p.grad is not None:
                        fisher[n] += p.grad.data ** 2 / n_samples
            return fisher

        def penalty(self, lam=1000):
            loss = 0
            for n, p in self.model.named_parameters():
                loss += (self.fisher[n] * (p - self.params[n]) ** 2).sum()
            return lam * loss


    # Train with EWC
    model_ewc = MLP()
    train_task(model_ewc, X_A, y_A, epochs=200)
    acc_A_ewc_before = eval_acc(model_ewc, X_A, y_A)

    ewc = EWC(model_ewc, (X_A, y_A))

    # Train on B with EWC penalty
    opt = optim.Adam(model_ewc.parameters(), lr=0.01)
    for _ in range(200):
        opt.zero_grad()
        loss = nn.CrossEntropyLoss()(model_ewc(X_B), y_B) + ewc.penalty(lam=500)
        loss.backward()
        opt.step()

    acc_A_ewc_after = eval_acc(model_ewc, X_A, y_A)
    acc_B_ewc = eval_acc(model_ewc, X_B, y_B)

    print("=== EWC vs Naive Fine-tuning ===")
    print(f"{'':30} {'Naive':>8} {'EWC':>8}")
    print(f"{'Task A acc before B':30} {acc_A_before:>8.2%} {acc_A_ewc_before:>8.2%}")
    print(f"{'Task A acc after B':30} {acc_A_after:>8.2%} {acc_A_ewc_after:>8.2%}")
    print(f"{'Task B acc':30} {acc_B_after:>8.2%} {acc_B_ewc:>8.2%}")

except (ImportError, NameError):
    print("PyTorch not available. EWC adds a Fisher-information-weighted penalty")
    print("to prevent overwriting weights important for previous tasks.")

## 6. Sustainability Connection

Continual learning is directly linked to **sustainable AI**:

| Approach | CO₂ cost | Notes |
|---|---|---|
| Full retraining from scratch | Very high | Wastes all previous computation |
| Fine-tuning (naive) | Medium | Fast but causes forgetting |
| Continual learning (EWC/replay) | Low | Adapts without full retraining |
| Online learning | Very low | One pass through data |

Training GPT-3 once emitted ~552 tonnes of CO₂. Continual learning approaches that avoid full retraining are essential for sustainable AI deployment.

## 7. Summary

### Key Takeaways
- **Data streams** are infinite sequences where the distribution can change over time
- **Concept drift** is when P(y|x) changes — models trained on old data become stale
- **Online learning** adapts naturally to drift; batch learning requires retraining
- **ADWIN** detects drift by testing for distribution changes in a sliding window
- **Catastrophic forgetting** occurs when neural networks overwrite old knowledge
- **EWC** prevents forgetting by penalising changes to important weights
- Continual learning is key to **sustainable AI** — avoiding expensive full retraining

### Next Steps
- **E08** — Continual and Lifelong Learning (deeper coverage of CL strategies)
- **E09** — Self-Supervised Learning (pre-training that transfers across tasks)
- **E14** — Efficient and Green AI (broader sustainability in AI)